Q1 — Exploring Customer Orders with SQL Aggregation and Joins
Step 0 — Load Data into SQLite Database

In [1]:
import pandas as pd
import sqlite3

# Load datasets
customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

# Create in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Load tables into database
customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")

print("Tables loaded successfully!")

Tables loaded successfully!


Task 1 — Aggregation and Grouping
SQL Query

In [2]:
query_task1 = """
SELECT
    CustomerID,
    COUNT(OrderID) AS order_count,
    SUM(Freight) AS total_freight,
    AVG(Freight) AS avg_freight
FROM orders
GROUP BY CustomerID
ORDER BY total_freight DESC
"""

task1_result = pd.read_sql_query(query_task1, conn)

# Display top 10 rows
task1_result.head(10)

,customerID,order_count,total_freight,avg_freight
0,SAVEA,31,6683.70,215.603226
1,ERNSH,30,6205.39,206.846333
2,QUICK,28,5605.63,200.201071
3,HUNGO,19,2755.24,145.012632
4,RATTC,18,2134.21,118.567222
5,QUEEN,13,1982.70,152.515385
6,FOLKO,19,1678.08,88.320000
7,BERGS,18,1559.52,86.640000
8,FRANK,15,1403.44,93.562667
9,MEREP,13,1394.22,107.247692


Groups orders by customer
Counts number of orders
Calculates total freight
Calculates average freight
Sorts customers by highest spending


Task 2 — WHERE vs HAVING
Query A — Using WHERE

In [3]:
query_A = """
SELECT
    CustomerID,
    COUNT(OrderID) AS high_freight_orders
FROM orders
WHERE Freight > 50
GROUP BY CustomerID
"""

result_A = pd.read_sql_query(query_A, conn)

result_A.head()

,customerID,high_freight_orders
0,ALFKI,2
1,ANTON,2
2,AROUT,2
3,BERGS,11
4,BLAUS,1


Query B — Using HAVING

In [4]:
query_B = """
SELECT
    CustomerID,
    SUM(Freight) AS total_freight
FROM orders
GROUP BY CustomerID
HAVING SUM(Freight) > 500
"""

result_B = pd.read_sql_query(query_B, conn)

result_B.head()

,customerID,total_freight
0,BERGS,1559.52
1,BLONP,623.66
2,BONAP,1357.87
3,BOTTM,793.95
4,EASTC,832.34


Markdown Explanation — WHERE vs HAVING

Query A uses WHERE to filter individual rows before grouping. Only orders with Freight greater than 50 are included in the aggregation.

Query B uses HAVING to filter groups after aggregation. It first calculates the total freight for each customer, then returns only those customers whose total exceeds 500.

Therefore, the two queries produce different results because one filters rows before grouping, while the other filters aggregated results after grouping.

Task 3 — JOIN and Aggregation
Query 1 — INNER JOIN

In [5]:
query_inner_join = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    SUM(o.Freight) AS total_freight
FROM customers c
INNER JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CompanyName, c.Country
ORDER BY total_freight DESC
"""

inner_result = pd.read_sql_query(query_inner_join, conn)

inner_result.head()

,companyName,country,order_count,total_freight
0,Save-a-lot Markets,USA,31,6683.70
1,Ernst Handel,Austria,30,6205.39
2,QUICK-Stop,Germany,28,5605.63
3,Hungry Owl All-Night Grocers,Ireland,19,2755.24
4,Rattlesnake Canyon Grocery,USA,18,2134.21


Query 2 — LEFT JOIN

In [6]:
query_left_join = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    COALESCE(SUM(o.Freight), 0) AS total_freight
FROM customers c
LEFT JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CompanyName, c.Country
ORDER BY total_freight DESC
"""

left_result = pd.read_sql_query(query_left_join, conn)

left_result.head()

,companyName,country,order_count,total_freight
0,Save-a-lot Markets,USA,31,6683.70
1,Ernst Handel,Austria,30,6205.39
2,QUICK-Stop,Germany,28,5605.63
3,Hungry Owl All-Night Grocers,Ireland,19,2755.24
4,Rattlesnake Canyon Grocery,USA,18,2134.21


:Markdown Explanation — INNER JOIN vs LEFT JOIN

The INNER JOIN query returns only customers who have placed at least one order because it matches records that exist in both tables.

The LEFT JOIN query returns all customers, including those who have not placed any orders. For customers without orders, the freight value appears as NULL or 0 because there are no matching records in the orders table.